# Week 4 — Calibration and Analysis

Covers:
1. σ estimation from simulated mid-price returns, and the calibration gap
2. Baseline 1-hour session — time-series plots
3. Multi-seed γ-sweep — PnL, Sharpe, drawdown, fill rate, adverse selection
4. Inventory profiles
5. PnL / inventory-risk frontier
6. Key findings

**Flow config** (fixed throughout):  
`λ_lim=5, λ_mkt=0.3, λ_cancel=0.3, max_qty=3, depth=20×20`

**Strategy config** (fixed except γ):  
`σ=0.86, k=1.5, T=3600, max_inventory=30, max_half_spread=8`

In [ ]:
import sys
sys.path.insert(0, '..')

import math
import statistics
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from lob_sim import Simulator, AvellanedaStoikov, metrics
from lob_sim.flow import PoissonFlow
from lob_sim.events import Side

plt.rcParams.update({
    'figure.figsize': (13, 4),
    'axes.grid': True,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

# ── Shared parameters ────────────────────────────────────────────────────────
FLOW_KW = dict(
    lambda_lim=5.0, lambda_mkt=0.3, lambda_cancel=0.3,
    min_qty=1, max_qty=3, sigma_ref=0.3,
)
SIM_KW = dict(
    duration=3600, snapshot_interval=30,
    initial_depth=20, initial_qty_per_level=20,
)
STRAT_KW = dict(
    sigma=0.86, k=1.5, T=3600,
    order_size=1, max_inventory=30, max_half_spread=8,
)
GAMMAS = [0.0005, 0.001, 0.002, 0.005, 0.01, 0.05]
N_SEEDS = 12

def make_run(gamma, seed):
    flow  = PoissonFlow(**FLOW_KW, seed=seed)
    strat = AvellanedaStoikov(gamma=gamma, **STRAT_KW)
    sim   = Simulator(flow=flow, strategy=strat, **SIM_KW)
    r     = sim.run()
    return r, strat

print('Setup OK')

## 1. σ estimation and calibration gap

The AS strategy assumes a mid-price volatility σ. We estimate σ̂ directly from
the simulated mid-price series using 1-second bucketed returns, then compare
to the assumed value. The gap tells us whether the strategy's spread is
appropriately sized for the actual price process.

In [ ]:
def estimate_sigma(mid_ts, bucket=1.0):
    """Estimate realized σ from 1-second bucketed mid-price returns."""
    bkt = {}
    for t, v in mid_ts:
        bkt[int(t / bucket)] = v
    keys = sorted(bkt)
    vals = [bkt[k] for k in keys]
    rets = [vals[i] - vals[i-1] for i in range(1, len(vals))]
    return statistics.stdev(rets) / math.sqrt(bucket) if len(rets) > 1 else float('nan')

# Run 20 seeds with gamma=0.001 just to measure the price process
sigma_estimates = []
for seed in range(20):
    r, _ = make_run(gamma=0.001, seed=seed)
    sigma_estimates.append(estimate_sigma(r.mid_ts))

sig_mean = statistics.mean(sigma_estimates)
sig_std  = statistics.stdev(sigma_estimates)
print(f'Assumed σ (strategy input):  {STRAT_KW["sigma"]:.4f} ticks/√s')
print(f'Realized σ̂ (20-seed mean):   {sig_mean:.4f}  ± {sig_std:.4f} ticks/√s')
print(f'Calibration gap:             {sig_mean / STRAT_KW["sigma"]:.1f}×')
print()
print('Interpretation:')
print(f'  The AS half-spread at t=0 is sized for σ={STRAT_KW["sigma"]:.2f}.')
print(f'  The book\'s actual price process has σ̂≈{sig_mean:.2f}, ~{sig_mean/STRAT_KW["sigma"]:.0f}× larger.')
print(f'  This means the strategy\'s spread is chronically too narrow for realized vol.')
print(f'  In live trading, σ would be re-estimated daily before the session.')

## 2. Baseline session — time-series plots

In [ ]:
r_base, strat_base = make_run(gamma=0.001, seed=0)

def downsample(ts_list, n=2000):
    if len(ts_list) <= n:
        return ts_list
    step = max(1, len(ts_list) // n)
    return ts_list[::step]

fig = plt.figure(figsize=(14, 11))
gs  = gridspec.GridSpec(4, 1, hspace=0.45)

# Mid price
ax1 = fig.add_subplot(gs[0])
ds  = downsample(r_base.mid_ts)
if ds:
    t_, v_ = zip(*ds)
    ax1.plot(np.array(t_)/3600, v_, lw=0.6, color='navy')
ax1.set_ylabel('Mid (ticks)')
ax1.set_title('Mid-price path  (γ=0.001, seed=0)')

# Spread
ax2 = fig.add_subplot(gs[1])
ds  = downsample([(t,s) for t,s in r_base.spread_ts if s is not None])
if ds:
    t_, v_ = zip(*ds)
    ax2.plot(np.array(t_)/3600, v_, lw=0.5, color='teal', alpha=0.8)
ax2.set_ylabel('Spread (ticks)')
ax2.set_title('BBO spread')

# Inventory
ax3 = fig.add_subplot(gs[2])
ds  = downsample(r_base.inventory_ts)
if ds:
    t_, v_ = zip(*ds)
    ax3.plot(np.array(t_)/3600, v_, lw=0.8, color='darkorange')
    ax3.axhline(0, color='k', lw=0.5, ls='--')
    ax3.axhline( STRAT_KW['max_inventory'], color='r', lw=0.8, ls=':', label='limit')
    ax3.axhline(-STRAT_KW['max_inventory'], color='r', lw=0.8, ls=':')
    ax3.legend(fontsize=8)
ax3.set_ylabel('Inventory')
ax3.set_title('Inventory')

# Cumulative PnL
ax4 = fig.add_subplot(gs[3])
ds  = downsample(r_base.pnl_ts)
if ds:
    t_, v_ = zip(*ds)
    ax4.plot(np.array(t_)/3600, v_, lw=0.8, color='crimson')
    ax4.axhline(0, color='k', lw=0.5, ls='--')
ax4.set_xlabel('Time (hours)')
ax4.set_ylabel('PnL (ticks)')
ax4.set_title('Cumulative mark-to-market PnL')

plt.suptitle('Avellaneda-Stoikov — 1-hour baseline session', fontsize=12, y=1.01)
plt.savefig('baseline_session.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Final PnL: {r_base.final_pnl:.1f} ticks   Inventory: {r_base.final_inventory}')

## 3. Multi-seed γ-sweep

Run each γ with N_SEEDS independent seeds.  Report mean ± std for each metric.

In [ ]:
sweep_raw = {}   # gamma -> list of per-seed metric dicts

for gamma in GAMMAS:
    rows = []
    for seed in range(N_SEEDS):
        r, strat = make_run(gamma, seed)

        inv_stats = metrics.inventory_profile(r.inventory_ts) or {}
        strat_oids = strat.all_submitted_ids
        adv = metrics.adverse_selection(r.trades, r.mid_ts, strat_oids)
        fr  = metrics.fill_rate(strat_oids, r.trades, strat.n_quotes_submitted)

        mid_vals = [v for _, v in r.mid_ts]

        rows.append(dict(
            pnl          = r.final_pnl,
            sharpe       = metrics.sharpe_ratio(r.pnl_ts, 60),
            drawdown     = metrics.max_drawdown(r.pnl_ts),
            inv_std      = inv_stats.get('std', float('nan')),
            inv_mean     = inv_stats.get('mean', float('nan')),
            time_flat    = inv_stats.get('time_flat', float('nan')),
            fill_rate    = fr,
            adv_sel      = adv.get('overall', float('nan')),
            n_fills      = adv.get('n_fills', 0),
            sigma_real   = estimate_sigma(r.mid_ts),
            mid_range    = (max(mid_vals) - min(mid_vals)) if mid_vals else 0,
        ))
    sweep_raw[gamma] = rows
    print(f'γ={gamma:.4f}  done ({N_SEEDS} seeds)', flush=True)

print('Sweep complete.')

In [ ]:
def agg(rows, key):
    vals = [r[key] for r in rows if not math.isnan(r[key])]
    if not vals:
        return float('nan'), float('nan')
    return statistics.mean(vals), statistics.stdev(vals) if len(vals) > 1 else 0.0

print(f"{'γ':>8}  {'PnL μ':>9} {'±':>2} {'σ':>7}  "
      f"{'Sharpe μ':>9}  {'MaxDD μ':>9}  "
      f"{'InvStd μ':>9}  {'FillRate μ':>10}  {'AdvSel μ':>9}")
print('─' * 90)

sweep_agg = {}
for gamma in GAMMAS:
    rows = sweep_raw[gamma]
    d = {k: agg(rows, k) for k in
         ['pnl','sharpe','drawdown','inv_std','fill_rate','adv_sel','time_flat']}
    sweep_agg[gamma] = d
    print(f"{gamma:8.4f}  "
          f"{d['pnl'][0]:9.1f} ± {d['pnl'][1]:7.1f}  "
          f"{d['sharpe'][0]:9.3f}  "
          f"{d['drawdown'][0]:9.1f}  "
          f"{d['inv_std'][0]:9.2f}  "
          f"{d['fill_rate'][0]:10.4f}  "
          f"{d['adv_sel'][0]:9.3f}")

## 4. PnL distributions across seeds

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=False)
axes = axes.flatten()

for ax, gamma in zip(axes, GAMMAS):
    pnls = [row['pnl'] for row in sweep_raw[gamma]]
    ax.hist(pnls, bins=min(N_SEEDS, 8), color='steelblue', alpha=0.8, edgecolor='white')
    ax.axvline(statistics.mean(pnls), color='crimson', lw=1.5, ls='--', label=f'mean={statistics.mean(pnls):.0f}')
    ax.axvline(0, color='k', lw=0.8, ls=':')
    ax.set_title(f'γ = {gamma}')
    ax.set_xlabel('Final PnL (ticks)')
    ax.legend(fontsize=8)

plt.suptitle('PnL distributions across seeds  (1-hour session)', fontsize=12)
plt.tight_layout()
plt.savefig('pnl_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Inventory profiles

In [ ]:
# Show inventory histogram for three representative γ values
show_gammas = [GAMMAS[0], GAMMAS[2], GAMMAS[-1]]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, gamma in zip(axes, show_gammas):
    # Aggregate inventory from all seeds
    all_inv_ts = []
    for seed in range(N_SEEDS):
        r, _ = make_run(gamma, seed)
        all_inv_ts.extend(r.inventory_ts)
    stats = metrics.inventory_profile(all_inv_ts)
    if stats and 'histogram' in stats:
        levels, fracs = zip(*stats['histogram'])
        ax.bar(levels, fracs, color='darkorange', alpha=0.8)
        ax.set_title(f'γ = {gamma}\nmean={stats["mean"]:.2f}  std={stats["std"]:.2f}')
        ax.set_xlabel('Inventory')
        ax.set_ylabel('Time fraction')

plt.suptitle('Inventory time-in-state distribution (pooled over seeds)', fontsize=12)
plt.tight_layout()
plt.savefig('inventory_profiles.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. PnL / inventory-risk frontier and metric sweeps

In [ ]:
pnl_means    = [sweep_agg[g]['pnl'][0]      for g in GAMMAS]
pnl_stds     = [sweep_agg[g]['pnl'][1]      for g in GAMMAS]
sharpe_means = [sweep_agg[g]['sharpe'][0]   for g in GAMMAS]
dd_means     = [sweep_agg[g]['drawdown'][0] for g in GAMMAS]
inv_stds     = [sweep_agg[g]['inv_std'][0]  for g in GAMMAS]
adv_means    = [sweep_agg[g]['adv_sel'][0]  for g in GAMMAS]
fr_means     = [sweep_agg[g]['fill_rate'][0] for g in GAMMAS]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Frontier: PnL vs inventory std
ax = axes[0, 0]
sc = ax.scatter(inv_stds, pnl_means, c=np.log10(GAMMAS), cmap='viridis', s=100, zorder=3)
ax.errorbar(inv_stds, pnl_means, yerr=pnl_stds, fmt='none', color='gray', alpha=0.5)
for i, g in enumerate(GAMMAS):
    ax.annotate(f'γ={g}', (inv_stds[i], pnl_means[i]), xytext=(4, 4),
                textcoords='offset points', fontsize=7)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_xlabel('Inventory std (contracts)')
ax.set_ylabel('Mean PnL (ticks)')
ax.set_title('PnL / Inventory-risk frontier')
plt.colorbar(sc, ax=ax, label='log₁₀(γ)')

# Sharpe vs gamma
ax = axes[0, 1]
ax.semilogx(GAMMAS, sharpe_means, 'bo-', lw=1.5)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_xlabel('γ')
ax.set_ylabel('Sharpe ratio (1-min)')
ax.set_title('Sharpe vs γ')

# PnL vs gamma
ax = axes[0, 2]
ax.semilogx(GAMMAS, pnl_means, 'ro-', lw=1.5)
ax.fill_between(GAMMAS,
    [m-s for m,s in zip(pnl_means, pnl_stds)],
    [m+s for m,s in zip(pnl_means, pnl_stds)],
    alpha=0.2, color='red')
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_xlabel('γ')
ax.set_ylabel('Mean PnL (ticks)')
ax.set_title('PnL vs γ  (±1 std band)')

# Max drawdown vs gamma
ax = axes[1, 0]
ax.semilogx(GAMMAS, dd_means, 'gs-', lw=1.5)
ax.set_xlabel('γ')
ax.set_ylabel('Max drawdown (ticks)')
ax.set_title('Max drawdown vs γ')

# Adverse selection vs gamma
ax = axes[1, 1]
ax.semilogx(GAMMAS, adv_means, 'm^-', lw=1.5)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_xlabel('γ')
ax.set_ylabel('Adverse selection (ticks)')
ax.set_title('Adverse selection vs γ\n(negative = picked off)')

# Fill rate vs gamma
ax = axes[1, 2]
ax.semilogx(GAMMAS, [f*100 for f in fr_means], 'c^-', lw=1.5)
ax.set_xlabel('γ')
ax.set_ylabel('Fill rate (%)')
ax.set_title('Fill rate vs γ')

plt.suptitle('Avellaneda-Stoikov γ-sweep  (12 seeds × 1 hour)', fontsize=12)
plt.tight_layout()
plt.savefig('gamma_sweep.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Key findings

Print a structured summary suitable for the research note.

In [ ]:
# Best gamma by Sharpe (ignoring NaN)
valid = [(g, sweep_agg[g]['sharpe'][0]) for g in GAMMAS
         if not math.isnan(sweep_agg[g]['sharpe'][0])]
best_g, best_sharpe = max(valid, key=lambda x: x[1])

print('=' * 60)
print('KEY FINDINGS — Week 4')
print('=' * 60)
print()
print('σ calibration')
print(f'  Assumed σ: {STRAT_KW["sigma"]:.4f} ticks/√s')
print(f'  Realized σ̂ (flow): {sig_mean:.3f} ± {sig_std:.3f} ticks/√s')
print(f'  Gap: {sig_mean/STRAT_KW["sigma"]:.1f}×  '
      f'(strategy spread chronically too narrow for realized vol)')
print()
print('γ-sweep summary')
for g in GAMMAS:
    d = sweep_agg[g]
    print(f'  γ={g:.4f}  '
          f'PnL={d["pnl"][0]:+8.1f}  '
          f'Sharpe={d["sharpe"][0]:6.3f}  '
          f'MaxDD={d["drawdown"][0]:8.1f}  '
          f'InvStd={d["inv_std"][0]:5.2f}  '
          f'AdvSel={d["adv_sel"][0]:+6.2f}')
print()
print(f'Best Sharpe: γ={best_g}  Sharpe={best_sharpe:.3f}')
print()
print('Adverse selection')
for g in GAMMAS:
    adv = sweep_agg[g]['adv_sel'][0]
    half_spread_approx = 0.5*g*STRAT_KW['sigma']**2*STRAT_KW['T'] + \
                         (1/g)*math.log(1+g/STRAT_KW['k'])
    half_spread_capped = min(half_spread_approx, STRAT_KW['max_half_spread'])
    print(f'  γ={g:.4f}  AdvSel={adv:+.3f} ticks  '
          f'half-spread≈{half_spread_capped:.2f} ticks  '
          f'edge coverage: {100*(half_spread_capped+adv)/half_spread_capped:.0f}%'
          if half_spread_capped > 0 else '')
print()
print('Interpretation:')
print('  Adverse selection is negative for all γ: the mid moves against us')
print('  after every fill. This is the Poisson flow\'s market orders acting')
print('  as informed traders. The strategy earns the spread on most fills but')
print('  loses a fraction of it to post-fill adverse price movement.')
print('  A larger γ widens the spread (better coverage) but reduces fill rate.')